# Crush Calculator Pro
A safer, cleaner version of the original calculator. Run all cells from top to bottom, then use the tabbed interface.


In [1]:
import ast
import math
import os
import re
from itertools import product
from pathlib import Path

import ipywidgets as widgets
from IPython.display import FileLink, clear_output, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats


def parse_values(text):
    """Safely parse one number, comma-separated numbers, or a list/tuple."""
    text = str(text).strip()
    if not text:
        return []
    candidate = text if text.startswith(("[", "(")) or "," not in text else "[" + text + "]"
    try:
        value = ast.literal_eval(candidate)
    except (SyntaxError, ValueError) as exc:
        raise ValueError("Enter a number or list, for example: 5, 5,6,7, or [5, 6, 7].") from exc
    value = value if isinstance(value, (list, tuple)) else [value]
    if not value or any(isinstance(x, bool) or not isinstance(x, (int, float)) for x in value):
        raise ValueError("Every entered value must be numeric.")
    values = [float(x) for x in value]
    if not all(np.isfinite(values)):
        raise ValueError("Values must be finite numbers.")
    return values


def from_b_values(b0_values, b1_values, widths, masses):
    rows = []
    for b0, b1, width, mass in product(b0_values, b1_values, widths, masses):
        if width <= 0 or mass <= 0:
            raise ValueError("Width and mass must be greater than zero.")
        if b1 == 0:
            raise ValueError("b1 must be nonzero because G = A^2 / B.")
        A = (mass / width) * b0 * b1
        B = (mass / width) * b1**2
        rows.append({"b0": b0, "b1": b1, "width": width, "mass": mass,
                     "A": A, "B": B, "G": A**2 / B})
    return pd.DataFrame(rows)


def from_AB_values(A_values, B_values, widths, masses):
    rows = []
    for A, B, width, mass in product(A_values, B_values, widths, masses):
        if width <= 0 or mass <= 0:
            raise ValueError("Width and mass must be greater than zero.")
        if B <= 0:
            raise ValueError("B must be greater than zero for a real-valued b1.")
        b1 = math.sqrt(B * width / mass)
        b0 = A * width / (mass * b1)
        rows.append({"b0": b0, "b1": b1, "width": width, "mass": mass,
                     "A": A, "B": B, "G": A**2 / B})
    return pd.DataFrame(rows)


def derive_values(C1_values, C2_values, V1_values, V2_values):
    rows = []
    for C1, C2, V1, V2 in product(C1_values, C2_values, V1_values, V2_values):
        if C1 == C2:
            raise ValueError("C1 and C2 must differ; the slope is otherwise undefined.")
        b1 = (V2 - V1) / (C2 - C1)
        b0 = V1 - b1 * C1
        rows.append({"C1": C1, "C2": C2, "V1": V1, "V2": V2, "b0": b0, "b1": b1})
    return pd.DataFrame(rows)


def unique_path(folder, stem, suffix):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{stem}{suffix}"
    index = 2
    while path.exists():
        path = folder / f"{stem}_{index}{suffix}"
        index += 1
    return path


In [2]:
APP_CSS = """
<style>
.cc-title{background:#12344d;color:#fff;padding:16px 18px;border-radius:10px;margin-bottom:10px}
.cc-title h2{margin:0;font-size:23px}.cc-title p{margin:5px 0 0;color:#dbeafe}
.cc-section{font-weight:700;color:#12344d;margin:12px 0 4px}
.cc-note{padding:10px 12px;background:#fff7ed;border-left:4px solid #d97706;margin:8px 0}
.cc-help{padding:14px;border:1px solid #d1d5db;border-radius:8px;margin:9px 0;background:#fff}
.cc-muted{color:#4b5563;font-size:13px}
</style>
"""

SPEED_UNITS = ["mph", "fps", "mps", "kmph"]
LENGTH_UNITS = ["inch", "feet", "yard", "mm", "cm", "meter"]
MASS_UNITS = ["slug", "kg", "lbs"]
B1_UNITS = [f"{speed}/{length}" for speed in SPEED_UNITS for length in ["ft", "in", "cm", "mm", "m"]]
MODES = ["Manual list", "Range", "Linspace"]


class NumberInput:
    def __init__(self, label, initial=""):
        self.label = label
        style = {"description_width": "65px"}
        layout = widgets.Layout(width="220px")
        self.mode = widgets.Dropdown(options=MODES, value="Manual list", description="Mode:", style=style, layout=layout)
        self.manual = widgets.Text(value=initial, description=f"{label}:", placeholder="5 or 5, 6, 7", style=style, layout=layout)
        self.start = widgets.FloatText(value=0, description="Start:", style=style, layout=layout)
        self.stop = widgets.FloatText(value=10, description="Stop:", style=style, layout=layout)
        self.step = widgets.FloatText(value=1, description="Step:", style=style, layout=layout)
        self.points = widgets.BoundedIntText(value=5, min=1, max=10000, description="Points:", style=style, layout=layout)
        self.inputs = widgets.HBox()
        self.mode.observe(self._render, names="value")
        self._render()
        self.widget = widgets.VBox([self.mode, self.inputs], layout=widgets.Layout(margin="2px 10px 7px 0"))

    def _render(self, *_):
        if self.mode.value == "Manual list":
            self.inputs.children = [self.manual]
        elif self.mode.value == "Range":
            self.inputs.children = [self.start, self.stop, self.step]
        else:
            self.inputs.children = [self.start, self.stop, self.points]

    def values(self, required=False):
        if self.mode.value == "Manual list":
            result = parse_values(self.manual.value)
        elif self.mode.value == "Range":
            if self.step.value == 0:
                raise ValueError(f"{self.label}: range step cannot be zero.")
            if (self.stop.value - self.start.value) * self.step.value < 0:
                raise ValueError(f"{self.label}: the step points away from the stop value.")
            result = np.arange(self.start.value, self.stop.value + self.step.value * 1e-10, self.step.value).tolist()
        else:
            result = np.linspace(self.start.value, self.stop.value, self.points.value).tolist()
        if required and not result:
            raise ValueError(f"{self.label} is required.")
        return result


class CrushCalculator:
    def __init__(self):
        self.coefficient_results = pd.DataFrame()
        self.derived_results = pd.DataFrame()
        self._build()

    @staticmethod
    def dropdown(options, value, description):
        return widgets.Dropdown(options=options, value=value, description=description,
                                style={"description_width": "initial"}, layout=widgets.Layout(width="235px"))

    @staticmethod
    def row(number_input, unit=None):
        children = [number_input.widget] + ([unit] if unit is not None else [])
        return widgets.HBox(children, layout=widgets.Layout(flex_flow="row wrap", align_items="center"))

    def _build(self):
        self.width = NumberInput("Width", "36, 37")
        self.mass = NumberInput("Mass", "2000")
        self.b0 = NumberInput("b0", "5")
        self.b1 = NumberInput("b1", "5, 5, 6")
        self.A = NumberInput("A", "")
        self.B = NumberInput("B", "")
        self.width_unit = self.dropdown(LENGTH_UNITS, "inch", "Width unit:")
        self.mass_unit = self.dropdown(MASS_UNITS, "lbs", "Mass unit:")
        self.b0_unit = self.dropdown(SPEED_UNITS, "mph", "b0 unit:")
        self.b1_unit = self.dropdown(B1_UNITS, "mph/in", "b1 unit:")
        self.method = widgets.ToggleButtons(options=[("From b0 and b1", "b"), ("From A and B", "AB")],
                                            value="b", description="Method:", style={"description_width": "initial"})
        self.unit_note = widgets.HTML()
        for control in [self.width_unit, self.mass_unit, self.b0_unit, self.b1_unit]:
            control.observe(self._update_units, names="value")
        self._update_units()
        self.calculate_button = widgets.Button(description="Calculate coefficients", button_style="success", icon="calculator")
        self.calculate_button.on_click(self._calculate)
        self.calculate_output = widgets.Output()

        self.C1 = NumberInput("C1", "1, 2")
        self.C2 = NumberInput("C2", "16, 17")
        self.V1 = NumberInput("V1", "6")
        self.V2 = NumberInput("V2", "35")
        self.crush_unit = self.dropdown(LENGTH_UNITS, "inch", "Crush unit:")
        self.speed_unit = self.dropdown(SPEED_UNITS, "mph", "Speed unit:")
        self.derive_button = widgets.Button(description="Derive b0 and b1", button_style="success", icon="line-chart")
        self.derive_button.on_click(self._derive)
        self.derive_output = widgets.Output()

        # Velocity vs. crush plotting controls
        self.plot_source = self.dropdown(["Coefficient results", "Derived results"], "Coefficient results", "Result set:")
        self.plot_crush_unit = self.dropdown(LENGTH_UNITS, "inch", "Crush x unit:")
        self.plot_speed_unit = self.dropdown(SPEED_UNITS, "mph", "Velocity unit:")
        self.plot_x_min = widgets.FloatText(value=0, description="x minimum:", style={"description_width":"initial"})
        self.plot_x_max = widgets.FloatText(value=30, description="x maximum:", style={"description_width":"initial"})
        self.plot_points = widgets.BoundedIntText(value=151, min=2, max=5000, description="Plot points:", style={"description_width":"initial"})
        self.plot_button = widgets.Button(description="Plot velocity vs. crush", button_style="info", icon="line-chart")
        self.plot_button.on_click(self._plot_velocity)
        self.plot_output = widgets.Output()

        self.stats_source = self.dropdown(["Coefficient results", "Derived results"], "Coefficient results", "Result set:")
        self.stats_columns = widgets.SelectMultiple(options=[], description="Columns:", rows=8, style={"description_width":"initial"})
        self.stats_source.observe(self._refresh_stats_columns, names="value")
        self.percentile = widgets.BoundedFloatText(value=50, min=0, max=100, step=.1, description="Percentile:", style={"description_width":"initial"})
        self.include_ci = widgets.Checkbox(value=False, description="Include 95% confidence interval")
        self.stats_button = widgets.Button(description="Run statistical query", button_style="info", icon="bar-chart")
        self.stats_button.on_click(self._statistics)
        self.stats_output = widgets.Output()

        self.export_source = self.dropdown(["Coefficient results", "Derived results"], "Coefficient results", "Result set:")
        self.export_columns = widgets.SelectMultiple(options=[], description="Columns:", rows=8, style={"description_width":"initial"})
        self.export_source.observe(self._refresh_export_columns, names="value")
        self.csv_option = widgets.Checkbox(value=True, description="Combined CSV")
        self.hist_option = widgets.Checkbox(value=False, description="Histogram PNG and bin CSV")
        self.list_option = widgets.Checkbox(value=False, description="Print comma-separated list")
        self.bins = widgets.BoundedIntText(value=10, min=1, max=500, description="Bins:")
        self.export_button = widgets.Button(description="Export selected columns", button_style="success", icon="download")
        self.export_button.on_click(self._export)
        self.export_output = widgets.Output()

        calculate_tab = widgets.VBox([
            widgets.HTML('<div class="cc-section">1. Choose a calculation method</div>'), self.method,
            widgets.HTML('<div class="cc-section">2. Enter required width and mass</div>'),
            self.row(self.width, self.width_unit), self.row(self.mass, self.mass_unit),
            widgets.HTML('<div class="cc-section">3. Enter b0 and b1, or A and B</div>'),
            self.row(self.b0, self.b0_unit), self.row(self.b1, self.b1_unit), self.row(self.A), self.row(self.B),
            self.unit_note, self.calculate_button, self.calculate_output])
        derive_tab = widgets.VBox([
            widgets.HTML('<div class="cc-section">Derive the line V = b0 + b1 x C from two points</div>'),
            widgets.HBox([self.crush_unit, self.speed_unit]), self.row(self.C1), self.row(self.V1),
            self.row(self.C2), self.row(self.V2), self.derive_button, self.derive_output])
        plot_tab = widgets.VBox([
            widgets.HTML('<div class="cc-section">Velocity vs. crush</div><div class="cc-muted">Plots V = b0 + b1x for every calculated combination. Units are shown on both axes and in every curve label.</div>'),
            self.plot_source, widgets.HBox([self.plot_crush_unit, self.plot_speed_unit]),
            widgets.HBox([self.plot_x_min, self.plot_x_max, self.plot_points]),
            self.plot_button, self.plot_output])
        statistics_tab = widgets.VBox([
            widgets.HTML('<div class="cc-section">Analyze a result set</div><div class="cc-muted">Run a calculation first, then select one or more columns.</div>'),
            self.stats_source, self.stats_columns, widgets.HBox([self.percentile, self.include_ci]), self.stats_button, self.stats_output])
        export_tab = widgets.VBox([
            widgets.HTML('<div class="cc-section">Export a result set</div><div class="cc-muted">Files are saved in an exports folder beside the notebook.</div>'),
            self.export_source, self.export_columns, widgets.HBox([self.csv_option, self.hist_option, self.list_option]),
            self.bins, self.export_button, self.export_output])
        help_tab = widgets.HTML(value=self._help())

        tabs = widgets.Tab(children=[calculate_tab, derive_tab, plot_tab, statistics_tab, export_tab, help_tab])
        for index, title in enumerate(["Coefficients", "Derive", "Velocity Plot", "Statistics", "Export", "Help"]):
            tabs.set_title(index, title)
        header = widgets.HTML(APP_CSS + '<div class="cc-title"><h2>Crush Calculator Pro</h2><p>Coefficient calculation, derivation, statistics, and export in one notebook.</p></div>')
        self.ui = widgets.VBox([header, tabs])
        self._refresh_stats_columns()
        self._refresh_export_columns()

    def _result(self, source):
        return self.coefficient_results if source == "Coefficient results" else self.derived_results

    def _refresh_stats_columns(self, *_):
        self.stats_columns.options = list(self._result(self.stats_source.value).columns)

    def _refresh_export_columns(self, *_):
        self.export_columns.options = list(self._result(self.export_source.value).columns)

    def _update_units(self, *_):
        mass = "slug converted from lb" if self.mass_unit.value == "lbs" else self.mass_unit.value
        A_unit = f"{mass} x {self.b0_unit.value} x {self.b1_unit.value} / {self.width_unit.value}"
        B_unit = f"{mass} x ({self.b1_unit.value})^2 / {self.width_unit.value}"
        self.unit_note.value = f'<div class="cc-note"><b>Output unit expressions</b><br>A: {A_unit}<br>B: {B_unit}<br><b>Important:</b> unit menus label values; they do not convert length or speed. Keep units compatible.</div>'

    def _calculate(self, *_):
        with self.calculate_output:
            clear_output()
            try:
                widths = self.width.values(True)
                masses = self.mass.values(True)
                if any(x <= 0 for x in widths + masses):
                    raise ValueError("Width and mass must be greater than zero.")
                if self.mass_unit.value == "lbs":
                    masses = [x / 32.174 for x in masses]
                if self.method.value == "b":
                    self.coefficient_results = from_b_values(self.b0.values(True), self.b1.values(True), widths, masses)
                else:
                    self.coefficient_results = from_AB_values(self.A.values(True), self.B.values(True), widths, masses)
                self._refresh_stats_columns(); self._refresh_export_columns()
                print(f"Calculated {len(self.coefficient_results):,} combination(s).")
                display(self.coefficient_results)
            except Exception as exc:
                print(f"Error: {exc}")

    def _derive(self, *_):
        with self.derive_output:
            clear_output()
            try:
                self.derived_results = derive_values(self.C1.values(True), self.C2.values(True), self.V1.values(True), self.V2.values(True))
                self._refresh_stats_columns(); self._refresh_export_columns()
                print(f"Derived {len(self.derived_results):,} combination(s). b0 is in {self.speed_unit.value}; b1 is in {self.speed_unit.value}/{self.crush_unit.value}.")
                display(self.derived_results)
            except Exception as exc:
                print(f"Error: {exc}")

    def _plot_velocity(self, *_):
        with self.plot_output:
            clear_output()
            df = self._result(self.plot_source.value)
            if df.empty:
                print("No results exist. Run a coefficient or derivation calculation first.")
                return
            if not {"b0", "b1"}.issubset(df.columns):
                print("The selected results do not contain b0 and b1.")
                return
            x_min, x_max = self.plot_x_min.value, self.plot_x_max.value
            if not np.isfinite([x_min, x_max]).all() or x_max <= x_min:
                print("x maximum must be greater than x minimum, and both must be finite.")
                return
            x = np.linspace(x_min, x_max, self.plot_points.value)
            crush_unit = self.plot_crush_unit.value
            speed_unit = self.plot_speed_unit.value
            slope_unit = f"{speed_unit}/{crush_unit}"
            fig, ax = plt.subplots(figsize=(9, 5.5))
            for index, row in df.reset_index(drop=True).iterrows():
                velocity = row["b0"] + row["b1"] * x
                details = [f"#{index + 1}", f"b0={row['b0']:.4g} {speed_unit}", f"b1={row['b1']:.4g} {slope_unit}"]
                if "width" in row and pd.notna(row["width"]):
                    details.append(f"w0={row['width']:.4g} {crush_unit}")
                if "mass" in row and pd.notna(row["mass"]):
                    mass_label = "slug" if self.mass_unit.value == "lbs" else self.mass_unit.value
                    details.append(f"m={row['mass']:.4g} {mass_label}")
                if "A" in row and pd.notna(row["A"]): details.append(f"A={row['A']:.4g}")
                if "B" in row and pd.notna(row["B"]): details.append(f"B={row['B']:.4g}")
                if "C1" in row and pd.notna(row["C1"]):
                    details.extend([f"C1={row['C1']:.4g} {crush_unit}", f"C2={row['C2']:.4g} {crush_unit}",
                                    f"V1={row['V1']:.4g} {speed_unit}", f"V2={row['V2']:.4g} {speed_unit}"])
                ax.plot(x, velocity, linewidth=1.8, label=" | ".join(details))
            ax.set_title("Velocity vs. Crush")
            ax.set_xlabel(f"Crush, x ({crush_unit})")
            ax.set_ylabel(f"Velocity, V ({speed_unit})")
            ax.grid(True, alpha=.3)
            ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=8, borderaxespad=0)
            fig.tight_layout()
            display(fig)
            plt.close(fig)
            print(f"Plotted {len(df):,} combination(s) using V = b0 + b1x. "
                  f"Interpretation assumes b0 is in {speed_unit} and b1 is in {slope_unit}; no unit conversion is performed.")

    def _statistics(self, *_):
        with self.stats_output:
            clear_output()
            df = self._result(self.stats_source.value)
            columns = list(self.stats_columns.value)
            if df.empty:
                print("No results exist. Run a calculation first."); return
            if not columns:
                print("Select at least one column."); return
            rows = []
            for column in columns:
                data = pd.to_numeric(df[column], errors="coerce").dropna()
                if data.empty:
                    continue
                n = len(data); mean = data.mean()
                row = {"column": column, "n": n, "mean": mean, "median": data.median(),
                       "std": data.std(ddof=1) if n > 1 else np.nan,
                       "variance": data.var(ddof=1) if n > 1 else np.nan,
                       "min": data.min(), "max": data.max(), "range": data.max()-data.min(),
                       "p25": np.percentile(data, 25), f"p{self.percentile.value:g}": np.percentile(data, self.percentile.value),
                       "p75": np.percentile(data, 75)}
                if self.include_ci.value:
                    if n > 1 and row["std"] > 0:
                        low, high = stats.t.interval(.95, df=n-1, loc=mean, scale=stats.sem(data))
                    else:
                        low, high = np.nan, np.nan
                    row.update({"ci95_low": low, "ci95_high": high})
                rows.append(row)
            display(pd.DataFrame(rows).round(6))
            if any(row["n"] < 10 for row in rows):
                print("Caution: at least one column has fewer than 10 observations; inferential results are unstable.")

    def _export(self, *_):
        with self.export_output:
            clear_output()
            df = self._result(self.export_source.value)
            columns = list(self.export_columns.value)
            if df.empty:
                print("No results exist. Run a calculation first."); return
            if not columns:
                print("Select at least one column."); return
            selected = df[columns]
            if not any([self.csv_option.value, self.hist_option.value, self.list_option.value]):
                print("Choose at least one export format."); return
            if self.csv_option.value:
                path = unique_path("exports", "crush_results", ".csv")
                selected.to_csv(path, index=False)
                display(FileLink(str(path))); print(f"Saved {path}")
            if self.hist_option.value:
                for column in columns:
                    data = pd.to_numeric(selected[column], errors="coerce").dropna()
                    if data.empty:
                        print(f"Skipped {column}: it has no numeric data."); continue
                    counts, edges = np.histogram(data, bins=self.bins.value)
                    stem = re.sub(r"[^A-Za-z0-9_-]+", "_", column)
                    image_path = unique_path("exports/histograms", f"{stem}_histogram", ".png")
                    data_path = unique_path("exports/histograms", f"{stem}_histogram_bins", ".csv")
                    fig, ax = plt.subplots()
                    ax.hist(data, bins=self.bins.value)
                    ax.set(title=f"Histogram of {column}", xlabel=column, ylabel="Frequency")
                    ax.grid(True, alpha=.25); fig.tight_layout(); fig.savefig(image_path, dpi=160); plt.close(fig)
                    pd.DataFrame({"bin_start": edges[:-1], "bin_end": edges[1:], "count": counts}).to_csv(data_path, index=False)
                    display(FileLink(str(image_path))); display(FileLink(str(data_path)))
            if self.list_option.value:
                for column in columns:
                    print(f"\n{column}\n" + ", ".join(map(str, selected[column].tolist())))

    @staticmethod
    def _help():
        return """
        <div class="cc-help"><h3>Quick start</h3><ol>
        <li>On <b>Coefficients</b>, choose whether you know b0 and b1 or A and B.</li>
        <li>Enter width, mass, and the selected coefficient pair.</li>
        <li>Choose Manual list, Range, or Linspace for each field, then calculate.</li>
        <li>Use Statistics or Export after results appear.</li></ol></div>
        <div class="cc-help"><h3>What each calculation does</h3>
        <p><b>From b0 and b1:</b> A = (m/w0)b0b1, B = (m/w0)b1^2, and G = A^2/B.</p>
        <p><b>From A and B:</b> b1 = sqrt(Bw0/m), b0 = A(w0/m)/b1, and G = A^2/B.</p>
        <p>w0 is crush-zone width and m is mass. Width and mass must be positive. B must be positive for the inverse calculation.</p></div>
        <div class="cc-help"><h3>Deriving b0 and b1</h3>
        <p>The Derive tab fits V = b0 + b1C through (C1,V1) and (C2,V2). It uses b1=(V2-V1)/(C2-C1) and b0=V1-b1C1. C1 and C2 cannot be equal.</p></div>
        <div class="cc-help"><h3>Entering multiple values</h3><ul>
        <li><b>Manual list:</b> 5, or 5, 6, 7, or [5, 6, 7].</li>
        <li><b>Range:</b> start, stop, step. Stop is included when it lies on the sequence.</li>
        <li><b>Linspace:</b> start, stop, and an exact count of evenly spaced points.</li>
        <li>Every combination is evaluated. Two widths and three b1 values produce six rows when other inputs have one value.</li>
        <li>For Excel, use <code>=TEXTJOIN(",",TRUE,A1:A5)</code> and paste the result into a Manual list field.</li></ul></div>
        <div class="cc-help"><h3>Units and assumptions</h3>
        <p><b>Use internally consistent units.</b> Menus label values but do not convert length or speed. To preserve the original notebook behavior, mass entered as lbs is divided by 32.174 and used as slugs. Confirm that this matches your engineering convention.</p>
        <p>This notebook performs arithmetic, not crash-reconstruction validation. Verify calibration sources, conventions, inputs, and applicable professional methods before relying on results.</p></div>
        <div class="cc-help"><h3>Velocity plot</h3>
        <p>The Velocity Plot tab draws <b>V = b0 + b1x</b> for every row in the selected result set. Choose the crush and velocity units that match the b0 and b1 data. The x-axis, y-axis, and each legend entry include units. The plot does not convert units, so b0 must use the selected velocity unit and b1 must use velocity-unit/crush-unit.</p></div>
        <div class="cc-help"><h3>Statistics and exports</h3>
        <p>Statistics include count, mean, median, sample standard deviation, variance, range, percentiles, and an optional Student-t 95% confidence interval for the mean. Treat n below 10 cautiously. Exports include a combined CSV, histogram PNG plus bin CSV, and a printed comma-separated list.</p></div>
        """


app = CrushCalculator()
display(app.ui)


In [3]:
# Basic calculation checks
_test1 = from_b_values([5], [5], [36], [2000 / 32.174])
assert len(_test1) == 1 and np.isfinite(_test1.loc[0, "G"])
_test2 = derive_values([1], [16], [6], [35])
assert np.isclose(_test2.loc[0, "b1"], 29 / 15)
assert np.isclose(_test2.loc[0, "b0"], 6 - 29 / 15)
print("Core calculation checks passed.")


Core calculation checks passed.
